# Kompletny przewodnik: Plotly (Express + Graph Objects)

Przewodnik referencyjny (Python 3.10+, plotly ≥ 6, pandas). Notebook jest odpowiednikiem przewodnika po matplotlib/seaborn i używa **tego samego zbioru danych**, więc łatwo porównać składnię. Każda komórka kodu jest samodzielna po wykonaniu sekcji **0**. Kod jest „inline" (bez funkcji), a w tabelach parametrów ✅ = wymagany, ⬜ = opcjonalny.

**Spis treści**

0. Setup, dane przykładowe
1. Jak działa Plotly: `px` vs `go`, układ, osie, hover, szablony
2. Kolory: skale kolorów i palety
3. Wykresy liniowe i obszarowe
4. Wykresy słupkowe (w tym wyróżnianie słupków)
5. Pie, donut, sunburst, treemap, funnel, waterfall, Sankey
6. Rozkłady: histogram, ECDF, KDE (zamiennik `distplot`), rug
7. Rozkłady w grupach: box, violin, strip, ridge plot
8. Zależności: scatter, bubble, trendline, marginesy (`jointplot`), `lmplot`
9. Dwa wymiary i wielowymiarowe: density heatmap/contour (zamiast hexbin), scatter matrix, parallel coordinates, 3D, radar
10. Macierze: heatmap, clustermap
11. Linie, cieniowanie, adnotacje, wyróżnianie elementów
12. Układy złożone: facety, `make_subplots`, dual axis, dashboard, KPI
13. Interaktywność: dropdown, slider, animacje, zaznaczanie
14. Mapy
15. Zapis, wydajność, pułapki
16. Ściągawka: Plotly vs seaborn

## 0. Setup i dane przykładowe

In [52]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import plotly

print(plotly.__version__)

# Szablon globalny (wygląd wszystkich wykresów)
pio.templates.default = "plotly_white"       # inne: "plotly", "plotly_dark", "ggplot2", "seaborn", "simple_white", "presentation", "none"

# Renderer (gdzie wyświetlać): w VS Code zwykle działa domyślny; jawnie np.:
# pio.renderers.default = "vscode"     # VS Code
# pio.renderers.default = "notebook"   # klasyczny Jupyter
# pio.renderers.default = "browser"    # otwiera w przeglądarce

6.0.1


> Szablon `"seaborn"` w Plotly naśladuje wygląd seaborn, jeśli chcesz zachować znajomy styl.

### Dane przykładowe: zbiór „sprzedaż" (te same co w przewodniku matplotlib/seaborn)

Kolumny: data/czas, kategorie nominalne (`region`, `channel`), kategoria uporządkowana (`segment`), int (`customer_age`, `units`), float (`income`, `unit_price`, `discount`, `revenue`, `delivery_days`), porządkowa 1–5 (`satisfaction`), bool (`is_returning`), tekst/ID (`order_id`), braki danych (`delivery_days`).

In [53]:
rng = np.random.default_rng(42)
n = 1500

df = pd.DataFrame({
    "order_id": [f"ORD-{i:05d}" for i in range(n)],
    "order_date": pd.Timestamp("2023-01-01") + pd.to_timedelta(rng.integers(0, 730, n), unit="D"),
    "region": rng.choice(["Północ", "Południe", "Wschód", "Zachód", "Centrum"], n, p=[.2, .15, .2, .2, .25]),
    "channel": rng.choice(["Online", "Sklep", "Telefon"], n, p=[.55, .35, .10]),
    "segment": pd.Categorical(
        rng.choice(["Basic", "Standard", "Premium"], n, p=[.45, .40, .15]),
        categories=["Basic", "Standard", "Premium"], ordered=True),
    "customer_age": np.clip(rng.normal(40, 12, n), 18, 80).round().astype(int),
    "is_returning": rng.random(n) < 0.4,
})

seg_mult = df["segment"].astype(str).map({"Basic": 1.0, "Standard": 1.6, "Premium": 2.8}).to_numpy()
df["income"] = (rng.lognormal(8.2, 0.35, n) * (1 + (df["customer_age"] - 18) / 120)).round(0)
df["units"] = rng.poisson(3, n) + 1
df["unit_price"] = (rng.gamma(6, 12, n) * seg_mult).round(2)
df["discount"] = np.clip(rng.beta(2, 12, n), 0, 0.5).round(3)
df["revenue"] = (df["units"] * df["unit_price"] * (1 - df["discount"])).round(2)
df["delivery_days"] = (rng.gamma(3, 1.2, n) + (df["channel"] == "Online").to_numpy() * 1.5).round(1)
df["satisfaction"] = np.clip(
    np.round(4.2 - 0.15 * df["delivery_days"] + 0.3 * (df["segment"] == "Premium").to_numpy() + rng.normal(0, 0.8, n)),
    1, 5).astype(int)

df.loc[rng.choice(n, 40, replace=False), "delivery_days"] = np.nan
df["month"] = df["order_date"].dt.to_period("M").dt.to_timestamp()
df["quarter"] = df["order_date"].dt.to_period("Q").astype(str)
df["year"] = df["order_date"].dt.year
df["weekday"] = df["order_date"].dt.day_name()
df["returning_int"] = df["is_returning"].astype(int)
df["satisfaction_cat"] = df["satisfaction"].astype(str)       # do kolorowania kategorycznego (nie ciągłego!)

df_month = (df.groupby(["month", "channel"], as_index=False)
              .agg(revenue=("revenue", "sum"), orders=("order_id", "size")))

corr = df.select_dtypes("number").drop(columns=["year", "returning_int"]).corr()
pivot = df.pivot_table(index="region", columns="segment", values="revenue",
                       aggfunc="mean", observed=True)

smp = df.sample(600, random_state=4)         # próbka do wykresów punktowych
df.head()

,order_id,order_date,region,channel,segment,customer_age,is_returning,income,units,unit_price,discount,revenue,delivery_days,satisfaction,month,quarter,year,weekday,returning_int,satisfaction_cat
0,ORD-00000,2023-03-07,Południe,Online,Basic,56,False,4448.0,6,83.68,0.125,439.32,7.0,3,2023-03-01,2023Q1,2023,Tuesday,0,3
1,ORD-00001,2024-07-18,Południe,Sklep,Standard,43,True,3649.0,6,172.89,0.074,960.58,2.8,3,2024-07-01,2024Q3,2024,Thursday,1,3
2,ORD-00002,2024-04-22,Południe,Online,Standard,41,False,4314.0,4,49.94,0.241,151.62,2.2,4,2024-04-01,2024Q2,2024,Monday,0,4
3,ORD-00003,2023-11-17,Wschód,Sklep,Standard,29,True,7406.0,5,74.66,0.081,343.06,3.0,4,2023-11-01,2023Q4,2023,Friday,1,4
4,ORD-00004,2023-11-13,Zachód,Telefon,Standard,40,False,6666.0,2,82.20,0.139,141.55,3.0,3,2023-11-01,2023Q4,2023,Monday,0,3


> Dane z Polars: `df_pl.to_pandas()`. Plotly Express od wersji 5.16+ przyjmuje też bezpośrednio ramki Polars/PyArrow (przez `narwhals`), ale konwersja do pandas jest najbezpieczniejsza.

## 1. Jak działa Plotly

### 1.1. Dwa poziomy API

| Poziom | Moduł | Kiedy |
|---|---|---|
| **Plotly Express** | `px.*` | 90% zadań: jedna funkcja = jeden wykres z danych w formacie *long* (tidy), odpowiednik seaborn |
| **Graph Objects** | `go.*` | Pełna kontrola: własne trace'y, wykresy nietypowe (Waterfall, Sankey, Indicator), złożone układy |

`px.*` zwraca obiekt `go.Figure`, więc możesz go **dalej modyfikować** metodami graph objects. To najważniejszy wzorzec pracy:

```
fig = px.scatter(...)                  # 1. szybki wykres z danych
fig.update_traces(...)                 # 2. styl serii (markery, linie, hover) — „co jest rysowane"
fig.update_layout(...)                 # 3. układ (tytuł, legenda, marginesy, szablon)
fig.update_xaxes(...), update_yaxes()  # 4. osie
fig.add_hline(...), add_annotation()   # 5. elementy dodatkowe
fig.show()
```

**Odpowiedniki pojęć:**

| seaborn / matplotlib | Plotly |
|---|---|
| `hue` | `color` |
| `style` | `symbol` (scatter), `line_dash` (line), `pattern_shape` (bar) |
| `size` | `size` |
| `col`, `row` | `facet_col`, `facet_row` (+ `facet_col_wrap`) |
| `palette` | `color_discrete_sequence` / `color_discrete_map` / `color_continuous_scale` |
| `order` | `category_orders={"kolumna": [...]}` |
| `ax` | brak: wykres to `Figure`, osie przez `update_xaxes` lub `make_subplots` |
| `plt.show()` | `fig.show()` |

### 1.2. Wspólne parametry funkcji `px.*`

| Parametr | Opis |
|---|---|
| `data_frame` ✅ | DataFrame (pierwszy argument pozycyjny) |
| `x`, `y` ✅ | Nazwy kolumn (większość wykresów) |
| `color` ⬜ | Kolumna kodowana kolorem (tekst/bool → kategorie; liczba → skala ciągła) |
| `symbol`, `line_dash`, `pattern_shape` ⬜ | Kodowanie kształtem |
| `size`, `size_max` ⬜ | Rozmiar punktów (bubble) |
| `facet_col`, `facet_row`, `facet_col_wrap` ⬜ | Panele; `facet_col_spacing`, `facet_row_spacing` |
| `hover_name`, `hover_data` ⬜ | Pogrubiony nagłówek tooltipa / dodatkowe pola (`hover_data={"discount": ":.1%"}`) |
| `text` ⬜ | Etykiety na elementach |
| `title`, `labels`, `template` ⬜ | Tytuł, słownik `{"col": "Etykieta"}`, szablon |
| `color_discrete_sequence`, `color_discrete_map` ⬜ | Kolory kategorii (lista / słownik) |
| `color_continuous_scale`, `range_color`, `color_continuous_midpoint` ⬜ | Skala ciągła |
| `category_orders` ⬜ | Kolejność kategorii `{"region": [...]}` |
| `log_x`, `log_y`, `range_x`, `range_y` ⬜ | Skale log / zakresy |
| `opacity` ⬜ | Przezroczystość |
| `animation_frame`, `animation_group` ⬜ | Animacje |
| `width`, `height` ⬜ | Rozmiar w px |

### 1.3. Pełny przykład formatowania jednego wykresu

In [54]:
monthly = df.groupby("month", as_index=False)["revenue"].sum()

fig = px.line(monthly, x="month", y="revenue", markers=True,
              labels={"month": "Miesiąc", "revenue": "Przychód [PLN]"},
              title="Przychód miesięczny")

# --- styl serii (trace)
fig.update_traces(line=dict(color="#1f77b4", width=3, dash="solid"),       # dash: solid, dot, dash, dashdot
                  marker=dict(size=9, color="white", line=dict(color="#1f77b4", width=2)),
                  hovertemplate="<b>%{x|%b %Y}</b><br>Przychód: %{y:,.0f} PLN<extra></extra>")

# --- układ
fig.update_layout(
    title=dict(text="<b>Przychód miesięczny</b>", x=0, xanchor="left", font=dict(size=18)),   # tytuł do lewej
    font=dict(family="Arial", size=12, color="#333"),
    height=450, width=900,
    margin=dict(l=60, r=30, t=70, b=50),
    hovermode="x unified",                    # jeden tooltip dla całego X; inne: "closest", "x", "y", False
    plot_bgcolor="white",
    showlegend=False,
)

# --- osie
fig.update_xaxes(title_text="Miesiąc", tickformat="%b\n%Y", dtick="M3", showgrid=False, tickangle=0)
fig.update_yaxes(title_text="Przychód [PLN]", tickformat=",.0f", rangemode="tozero",
                 gridcolor="rgba(0,0,0,0.1)", zeroline=False, showline=False)

# --- elementy dodatkowe
fig.add_hline(y=monthly["revenue"].mean(), line_dash="dash", line_color="crimson",
              annotation_text=f"Średnia: {monthly['revenue'].mean():,.0f}", annotation_position="top left")
peak = monthly.loc[monthly["revenue"].idxmax()]
fig.add_annotation(x=peak["month"], y=peak["revenue"], text=f"Max: {peak['revenue']:,.0f}",
                   showarrow=True, arrowhead=2, arrowcolor="black", ax=0, ay=-40)
fig.show()

### 1.4. Ściągawka: metody `Figure`

| Cel | Metoda | Ważne parametry |
|---|---|---|
| Styl serii | `fig.update_traces(...)` | `selector=dict(name="Online")` zmienia tylko wybraną serię; `row=`, `col=`; `marker`, `line`, `opacity`, `hovertemplate`, `textposition`, `mode` |
| Układ | `fig.update_layout(...)` | `title`, `font`, `width`/`height`, `margin`, `legend`, `template`, `hovermode`, `barmode`, `bargap`, `showlegend`, `colorway`, `coloraxis_colorbar` |
| Osie | `fig.update_xaxes(...)`, `update_yaxes(...)` | `title_text`, `range`, `type` (`"log"`, `"date"`, `"category"`), `tickformat`, `tickprefix/ticksuffix`, `dtick`, `tickangle`, `showgrid`, `gridcolor`, `zeroline`, `autorange="reversed"`, `matches`, `rangeslider_visible` |
| Legenda | `fig.update_layout(legend=dict(...))` | `orientation="h"`, `x`, `y`, `xanchor`, `yanchor`, `title_text`, `font_size`, `bgcolor` |
| Hover | `update_traces(hovertemplate=...)` | `%{x}`, `%{y:,.2f}`, `%{customdata[0]}`, `<extra></extra>` (usuwa ramkę z nazwą serii) |
| Dodanie serii | `fig.add_trace(go.Scatter(...))` | `row=`, `col=`, `secondary_y=` |
| Linie/pasy | `add_hline`, `add_vline`, `add_hrect`, `add_vrect`, `add_shape` | zob. sekcja 11 |
| Tekst | `add_annotation` | `x`, `y`, `text`, `showarrow`, `arrowhead`, `ax`, `ay`, `xref`, `yref` (`"paper"` = 0–1 w obrębie wykresu) |

**Legenda pozioma nad wykresem** (najczęściej wymagana poprawka):

In [55]:
fig = px.line(df_month, x="month", y="revenue", color="channel")
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1, title_text=""))
fig.show()

**Selektor serii:** zmiana tylko jednej serii.

In [56]:
fig = px.line(df_month, x="month", y="revenue", color="channel")
fig.update_traces(line=dict(width=4, color="#d62728"), selector=dict(name="Online"))                 # wyróżnij Online
fig.update_traces(line=dict(color="lightgray"), selector=lambda t: t.name != "Online")               # reszta szara
fig.show()

### 1.5. Formatowanie liczb, dat i hover

| Format d3 | Efekt |
|---|---|
| `",.0f"` | 12,345 |
| `".2f"` | 12.35 |
| `".1%"` | 12.3% |
| `"$,.0f"` | $12,345 |
| `".2s"` | 12k (notacja SI) |
| `"%Y-%m-%d"`, `"%b %Y"`, `"%d.%m.%Y"` | formaty dat (`tickformat`/hover) |

In [57]:
fig = px.scatter(smp, x="unit_price", y="revenue", color="segment",
                 hover_name="order_id",                                             # pogrubiony nagłówek
                 hover_data={"discount": ":.1%", "units": True, "unit_price": ":.2f", "segment": False},
                 labels={"unit_price": "Cena jedn.", "revenue": "Przychód"})
fig.update_xaxes(tickprefix="PLN ", tickformat=",.0f")
fig.update_yaxes(tickformat=".2s")
fig.show()

### 1.6. Zmiana szablonu i własny motyw

In [58]:
fig = px.histogram(df, x="revenue", nbins=40, template="seaborn", title="Szablon: seaborn")
fig.show()

# własny szablon, np. wspólny wygląd w całej firmie
custom = go.layout.Template(layout=dict(
    font=dict(family="Arial", size=12), colorway=["#264653", "#2a9d8f", "#e9c46a", "#f4a261", "#e76f51"],
    plot_bgcolor="white", xaxis=dict(showgrid=False), yaxis=dict(gridcolor="#eee")))
pio.templates["firmowy"] = custom
fig = px.histogram(df, x="revenue", color="segment", nbins=40, template="firmowy")
fig.show()

## 2. Kolory

| Rodzaj danych | Parametr | Przykłady |
|---|---|---|
| Kategorie | `color_discrete_sequence`, `color_discrete_map` | `px.colors.qualitative.Set2`, `Plotly`, `D3`, `Safe`, `Pastel`, `Bold` |
| Wartości ciągłe | `color_continuous_scale` | `"Viridis"`, `"Plasma"`, `"Blues"`, `"YlGnBu"`, `"Turbo"` |
| Z neutralnym środkiem | `color_continuous_scale` + `color_continuous_midpoint=0` | `"RdBu_r"`, `"Picnic"`, `"Spectral"`, `"balance"`, `"PiYG"` |

Odwrócenie skali: dodaj `_r` (`"Viridis_r"`). Kolor zależy od **typu kolumny**: kolumna liczbowa → skala ciągła, tekst/bool/kategoria → dyskretne. Kolumnę liczbową o kilku wartościach (np. `satisfaction`) zamień na tekst (`satisfaction_cat`), jeśli ma być kolorowana kategorycznie.

In [59]:
# podgląd palet (swatches)
px.colors.qualitative.swatches().show()
px.colors.sequential.swatches_continuous().show()
px.colors.diverging.swatches_continuous().show()

In [60]:
# stałe kolory kategorii w całym raporcie (zalecane)
cmap = {"Online": "#1f77b4", "Sklep": "#ff7f0e", "Telefon": "#7f7f7f"}
fig = px.bar(df.groupby("channel", as_index=False)["revenue"].sum(), x="channel", y="revenue",
             color="channel", color_discrete_map=cmap)

# próbkowanie koloru ze skali ciągłej (np. dla własnych trace'ów)
colors = px.colors.sample_colorscale("Viridis", [0, 0.25, 0.5, 0.75, 1])
print(colors)

# własna skala ciągła: lista [pozycja 0–1, kolor]
custom_scale = [[0, "#ffffff"], [0.5, "#8ecae6"], [1, "#023047"]]
fig2 = px.density_heatmap(df, x="unit_price", y="revenue", color_continuous_scale=custom_scale)
fig.show(); fig2.show()

['rgb(68, 1, 84)', 'rgb(59, 81, 138)', 'rgb(34, 144, 140)', 'rgb(96, 200, 96)', 'rgb(253, 231, 37)']


## 3. Wykresy liniowe i obszarowe

**Kiedy:** szeregi czasowe, trendy. **Interpretacja:** nachylenie = tempo zmiany, sezonowość = powtarzalne wzorce. W Plotly **nie ma** automatycznego przedziału ufności jak w `sns.lineplot`: agregujesz dane sam (pandas/SQL/Polars) i opcjonalnie dodajesz pasmo ręcznie.

### 3.1. `px.line`: parametry

| Parametr | Opis |
|---|---|
| `data_frame`, `x`, `y` ✅ | Dane i kolumny |
| `color`, `line_dash`, `symbol` ⬜ | Kodowanie serii kolorem / stylem linii / znacznikiem |
| `line_group` ⬜ | Kolumna grupująca osobne linie bez rozróżniania kolorem |
| `markers` ⬜ | `True` pokazuje punkty |
| `line_shape` ⬜ | `"linear"`, `"spline"`, `"hv"`, `"vh"`, `"hvh"` (schodkowy, np. dla zmian stanu) |
| `error_y`, `error_y_minus` ⬜ | Kolumny z błędami (słupki błędów) |
| `text` ⬜ | Etykiety punktów |
| `facet_col`, `facet_row`, `facet_col_wrap` ⬜ | Panele |
| `animation_frame` ⬜ | Animacja |
| `render_mode` ⬜ | `"auto"`, `"svg"`, `"webgl"` (szybszy dla > 1000 punktów) |
| `range_x`, `range_y`, `log_y`, `labels`, `title`, `hover_data` ⬜ | |

In [61]:
# a) wiele serii + kolory stałe + znaczniki
fig = px.line(df_month, x="month", y="revenue", color="channel", markers=True,
              color_discrete_map={"Online": "#1f77b4", "Sklep": "#ff7f0e", "Telefon": "#7f7f7f"},
              line_dash="channel", labels={"revenue": "Przychód", "month": ""}, title="Przychód wg kanału")
fig.update_traces(line_width=2.5, marker_size=7)
fig.update_layout(hovermode="x unified", legend=dict(orientation="h", y=1.1, x=0, title_text=""))
fig.show()

# b) linia schodkowa i spline
agg = df.groupby("month", as_index=False)["revenue"].sum()
fig = go.Figure()
fig.add_trace(go.Scatter(x=agg["month"], y=agg["revenue"], mode="lines", line=dict(shape="hv", color="#4c72b0"), name="schodkowa (hv)"))
fig.add_trace(go.Scatter(x=agg["month"], y=agg["revenue"] * 0.8, mode="lines", line=dict(shape="spline", smoothing=1, color="#e76f51"), name="spline"))
fig.show()

### 3.2. Średnia + pasmo niepewności (odpowiednik `lineplot` z CI)

Ręcznie: agregacja z błędem standardowym i `go.Scatter` z `fill="toself"`/`"tonexty"`.

In [62]:
g = df.groupby("month")["revenue"].agg(["mean", "std", "count"]).reset_index()
g["sem"] = g["std"] / np.sqrt(g["count"])
g["lo"] = g["mean"] - 1.96 * g["sem"]
g["hi"] = g["mean"] + 1.96 * g["sem"]

fig = go.Figure()
fig.add_trace(go.Scatter(x=g["month"], y=g["hi"], mode="lines", line=dict(width=0), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=g["month"], y=g["lo"], mode="lines", line=dict(width=0), fill="tonexty",   # wypełnia do poprzedniej serii
                         fillcolor="rgba(42,157,143,0.25)", name="95% CI", hoverinfo="skip"))
fig.add_trace(go.Scatter(x=g["month"], y=g["mean"], mode="lines+markers", line=dict(color="#2a9d8f", width=3), name="Średnia"))
fig.update_layout(title="Średni przychód zamówienia + 95% CI", hovermode="x unified")
fig.show()

# alternatywa: słupki błędów w px (error_y)
fig = px.line(g, x="month", y="mean", error_y=1.96 * g["sem"], markers=True, title="Słupki błędów (error_y)")
fig.show()

### 3.3. Średnia krocząca i range slider

In [63]:
agg = df.groupby("month", as_index=False)["revenue"].sum()
agg["roll3"] = agg["revenue"].rolling(3, center=True).mean()

fig = go.Figure()
fig.add_trace(go.Scatter(x=agg["month"], y=agg["revenue"], mode="lines+markers", name="Dane", line=dict(color="lightgray"), marker=dict(color="gray")))
fig.add_trace(go.Scatter(x=agg["month"], y=agg["roll3"], mode="lines", name="Średnia krocząca 3m", line=dict(color="#1f77b4", width=3)))
fig.update_xaxes(
    rangeslider_visible=True,                                    # suwak zakresu pod wykresem
    rangeselector=dict(buttons=[                                 # przyciski szybkiego zakresu
        dict(count=3, label="3m", step="month", stepmode="backward"),
        dict(count=6, label="6m", step="month", stepmode="backward"),
        dict(count=1, label="YTD", step="year", stepmode="todate"),
        dict(step="all", label="Całość")]))
fig.update_layout(title="Range slider + selector", height=500)
fig.show()

### 3.4. Wykres obszarowy: `px.area`

**Kiedy:** skład całości w czasie. Parametr `groupnorm` zamienia go na udziały.

| Parametr | Opis |
|---|---|
| `x`, `y`, `color` ⬜ | Standardowo |
| `groupnorm` ⬜ | `"fraction"` lub `"percent"` (normalizacja do 100% w każdym X) |
| `line_group`, `pattern_shape`, `markers` ⬜ | |

In [64]:
fig = px.area(df_month, x="month", y="revenue", color="channel", color_discrete_sequence=px.colors.qualitative.Set2,
              title="Stacked area")
fig.show()

fig = px.area(df_month, x="month", y="revenue", color="channel", groupnorm="percent",
              color_discrete_sequence=px.colors.qualitative.Set2, title="Udział kanałów (100%)")
fig.update_yaxes(ticksuffix="%")
fig.show()

## 4. Wykresy słupkowe

**Kiedy:** porównanie wartości między kategoriami. Oś Y od 0. **Ważne:** `px.bar` **nie agreguje**: jeśli podasz surowe dane, słupki zostaną „nałożone" (zsumowane wizualnie). Zawsze agreguj przez `groupby` albo użyj `px.histogram(histfunc=...)` jako szybkiej agregacji.

### 4.1. `px.bar`: parametry

| Parametr | Opis |
|---|---|
| `x`, `y` ✅ | Kategoria i wartość (`orientation="h"` + zamiana osi = słupki poziome) |
| `color` ⬜ | Podział na serie |
| `barmode` ⬜ | `"relative"` (domyślnie, stos), `"group"` (obok siebie), `"overlay"`, `"stack"` |
| `orientation` ⬜ | `"v"` / `"h"` |
| `text`, `text_auto` ⬜ | Etykiety: `text_auto=True` lub format `".2s"`, `",.0f"` |
| `pattern_shape` ⬜ | Deseń (hatch) wg kategorii |
| `base` ⬜ | Dolna granica słupka (np. wykres „floating") |
| `error_y` ⬜ | Słupki błędów |
| `opacity`, `color_discrete_map`, `category_orders` ⬜ | |

In [65]:
tot = df.groupby("region", as_index=False)["revenue"].sum().sort_values("revenue", ascending=False)

# a) podstawowy, posortowany, z etykietami
fig = px.bar(tot, x="region", y="revenue", text_auto=",.0f", color_discrete_sequence=["#4c72b0"],
             category_orders={"region": tot["region"].tolist()}, title="Przychód wg regionu")
fig.update_traces(textposition="outside", cliponaxis=False)      # etykiety nad słupkami
fig.update_layout(yaxis_tickformat=",.0f", bargap=0.25)
fig.show()

# b) grupowane (barmode='group') i skumulowane
seg = df.groupby(["region", "segment"], as_index=False, observed=True)["revenue"].sum()
fig = px.bar(seg, x="region", y="revenue", color="segment", barmode="group",
             color_discrete_sequence=px.colors.sequential.Viridis[::3], category_orders={"segment": ["Basic", "Standard", "Premium"]})
fig.update_layout(bargap=0.2, bargroupgap=0.05)                  # odstęp między grupami / w grupie
fig.show()

fig = px.bar(seg, x="region", y="revenue", color="segment", barmode="relative", text_auto=".2s")      # skumulowane
fig.show()

# c) 100% stacked: normalizacja słupków
fig = px.histogram(df, x="region", color="segment", barnorm="percent", text_auto=".0f",
                   category_orders={"segment": ["Basic", "Standard", "Premium"]}, title="Udział segmentów w regionie (%)")
fig.update_layout(yaxis_title="%")
fig.show()

# d) poziome
fig = px.bar(tot.sort_values("revenue"), x="revenue", y="region", orientation="h", text_auto=",.0f")
fig.show()

### 4.2. Wyróżnianie wybranych słupków

In [66]:
# a) przez kolumnę pomocniczą + color_discrete_map (zalecane, działa też z legendą)
tot["wyroznij"] = np.where(tot["revenue"] == tot["revenue"].max(), "Najlepszy", "Pozostałe")
fig = px.bar(tot, x="region", y="revenue", color="wyroznij", text_auto=",.0f",
             color_discrete_map={"Najlepszy": "#d62728", "Pozostałe": "#b8c0cc"})
fig.update_layout(showlegend=False)
fig.add_hline(y=tot["revenue"].mean(), line_dash="dash", annotation_text="średnia", annotation_position="top right")
fig.show()

# b) lista kolorów w marker_color (kolejność = kolejność słupków)
fig = px.bar(tot, x="region", y="revenue")
fig.update_traces(marker_color=["#d62728" if v > tot["revenue"].mean() else "#b8c0cc" for v in tot["revenue"]],
                  marker_line_color="black", marker_line_width=1.2)
fig.show()

# c) deseń (hatch) dla wybranych słupków i przezroczystość
fig = px.bar(tot, x="region", y="revenue")
fig.update_traces(marker=dict(color="#8da0cb", pattern=dict(shape=["/" if v < tot["revenue"].mean() else "" for v in tot["revenue"]])))
fig.show()

### 4.3. Słupki z przedziałem ufności i lollipop

In [67]:
s = df.groupby("region")["revenue"].agg(["mean", "std", "count"]).reset_index()
s["ci"] = 1.96 * s["std"] / np.sqrt(s["count"])
fig = px.bar(s, x="region", y="mean", error_y="ci", color_discrete_sequence=["#8da0cb"], title="Średnia + 95% CI")
fig.show()

# lollipop: linie + punkty (go)
s = df.groupby("region")["revenue"].mean().sort_values()
fig = go.Figure()
for reg, val in s.items():
    fig.add_shape(type="line", x0=0, x1=val, y0=reg, y1=reg, line=dict(color="#8da0cb", width=3), layer="below")
fig.add_trace(go.Scatter(x=s.values, y=s.index, mode="markers", marker=dict(size=14, color="#264653"), name=""))
fig.update_layout(title="Lollipop: średni przychód", xaxis_rangemode="tozero", showlegend=False)
fig.show()

## 5. Udziały i hierarchie: pie, donut, sunburst, treemap, funnel, waterfall, Sankey

### 5.1. Pie / donut: `px.pie`

**Kiedy:** udział części w całości przy ≤ 5 kategoriach o wyraźnie różnych wartościach (przy wielu kategoriach: słupki 100%).

| Parametr | Opis |
|---|---|
| `names`, `values` ✅ | Kolumna etykiet, kolumna wartości |
| `hole` ⬜ | 0–1; np. `0.4` = donut |
| `color`, `color_discrete_map` ⬜ | Kolory wycinków |
| `hover_data`, `title` ⬜ | |

`update_traces`: `textinfo` (`"percent"`, `"label"`, `"value"`, kombinacje `"percent+label"`), `textposition` (`"inside"`, `"outside"`, `"auto"`), `pull` (lista wysunięć), `rotation`, `direction`, `sort`, `marker=dict(line=dict(color="white", width=2))`.

In [68]:
ch = df.groupby("channel", as_index=False)["revenue"].sum()

fig = px.pie(ch, names="channel", values="revenue", color="channel", color_discrete_map=cmap, title="Pie z wysuniętym wycinkiem")
fig.update_traces(textinfo="percent+label", pull=[0.08, 0, 0], textposition="inside",
                  marker=dict(line=dict(color="white", width=2)))
fig.show()

fig = px.pie(ch, names="channel", values="revenue", hole=0.5, color="channel", color_discrete_map=cmap)
fig.update_traces(textinfo="percent", rotation=90)
fig.add_annotation(text=f"<b>{ch['revenue'].sum()/1e3:,.0f} tys.</b><br>PLN", x=0.5, y=0.5, showarrow=False, font_size=16)   # opis w środku
fig.update_layout(title="Donut z podsumowaniem")
fig.show()

### 5.2. Sunburst, treemap, icicle: hierarchie

**Kiedy:** udział w całości **na kilku poziomach hierarchii** (region → segment → kanał). Treemap czytelniejszy od sunburst (pola prostokątne łatwiej porównać). Parametry: `path` ✅ (lista kolumn od korzenia), `values` ⬜, `color` ⬜ (kolor wg średniej ważonej lub kategorii), `color_continuous_scale`, `maxdepth`, `branchvalues`.

In [69]:
fig = px.sunburst(df, path=["region", "segment", "channel"], values="revenue", color="region",
                  color_discrete_sequence=px.colors.qualitative.Set2, maxdepth=3, title="Sunburst")
fig.update_traces(textinfo="label+percent parent")
fig.show()

fig = px.treemap(df, path=[px.Constant("Cała sprzedaż"), "region", "segment"], values="revenue",
                 color="discount", color_continuous_scale="RdYlGn_r", title="Treemap: wielkość = przychód, kolor = średni rabat")
fig.update_traces(root_color="lightgray", textinfo="label+value+percent root")
fig.update_layout(margin=dict(t=50, l=10, r=10, b=10))
fig.show()

### 5.3. Funnel (lejek) i waterfall (wodospad)

In [70]:
# funnel: etapy procesu (tu: sztuczny lejek konwersji)
stages = pd.DataFrame({"etap": ["Wizyty", "Koszyk", "Płatność", "Zakup"], "liczba": [10000, 3200, 1500, 900]})
fig = px.funnel(stages, x="liczba", y="etap", title="Lejek konwersji")
fig.update_traces(textinfo="value+percent initial")
fig.show()

# waterfall: wkład składników w zmianę wyniku (graph_objects)
fig = go.Figure(go.Waterfall(
    x=["Start", "Online", "Sklep", "Telefon", "Zwroty", "Koniec"],
    measure=["absolute", "relative", "relative", "relative", "relative", "total"],
    y=[100, 45, 30, 8, -22, None], text=["100", "+45", "+30", "+8", "-22", ""],
    increasing=dict(marker_color="#2a9d8f"), decreasing=dict(marker_color="#e76f51"), totals=dict(marker_color="#264653"),
    connector=dict(line=dict(color="gray", dash="dot"))))
fig.update_layout(title="Waterfall: zmiana przychodu", showlegend=False)
fig.show()

### 5.4. Sankey: przepływy między kategoriami

In [71]:
flow = df.groupby(["channel", "segment"], observed=True, as_index=False)["revenue"].sum()
nodes = list(flow["channel"].unique()) + [s for s in flow["segment"].astype(str).unique()]
idx = {name: i for i, name in enumerate(nodes)}

fig = go.Figure(go.Sankey(
    node=dict(label=nodes, pad=20, thickness=18, color="#8da0cb"),
    link=dict(source=flow["channel"].map(idx), target=flow["segment"].astype(str).map(idx), value=flow["revenue"],
              color="rgba(141,160,203,0.4)")))
fig.update_layout(title="Przepływ przychodu: kanał → segment")
fig.show()

## 6. Rozkłady zmiennej ciągłej

### 6.1. Histogram: `px.histogram`

**Kiedy:** kształt rozkładu. **Interpretacja:** wysokość = liczność (lub inna normalizacja/agregacja). Wygląd zależy od binów, więc sprawdź kilka ustawień (`nbins`).

| Parametr | Opis |
|---|---|
| `x` (lub `y`) ✅ | Zmienna; podanie `y` + `histfunc` pozwala agregować (np. suma przychodu po kategorii) |
| `nbins` ⬜ | Przybliżona liczba binów (dokładny rozmiar: `fig.update_traces(xbins=dict(start=, end=, size=))`) |
| `histfunc` ⬜ | `"count"` (domyślnie), `"sum"`, `"avg"`, `"min"`, `"max"` |
| `histnorm` ⬜ | `"percent"`, `"probability"`, `"density"`, `"probability density"` |
| `cumulative` ⬜ | `True` = dystrybuanta (histogram skumulowany) |
| `color` ⬜ | Grupy |
| `barmode` ⬜ | `"stack"`, `"overlay"`, `"group"`, `"relative"` |
| `barnorm` ⬜ | Normalizacja słupków do 100% (`"percent"`) |
| `marginal` ⬜ | `"rug"`, `"box"`, `"violin"`: dodatkowy wykres na marginesie |
| `opacity`, `range_x`, `log_x`, `text_auto`, `facet_col` ⬜ | |

In [72]:
# a) podstawowy z markerami brzegowymi (box) i liniami mediany/średniej
fig = px.histogram(df, x="revenue", nbins=40, marginal="box", color_discrete_sequence=["#4c72b0"], title="Histogram + box na marginesie")
fig.add_vline(x=df["revenue"].mean(), line_dash="dash", line_color="red", annotation_text="średnia")
fig.add_vline(x=df["revenue"].median(), line_dash="dot", line_color="black", annotation_text="mediana", annotation_position="bottom right")
fig.show()

# b) grupy, nakładanie z przezroczystością, normalizacja do gęstości
fig = px.histogram(df, x="revenue", color="segment", nbins=50, barmode="overlay", opacity=0.55,
                   histnorm="probability density", color_discrete_sequence=px.colors.sequential.Viridis[::3],
                   category_orders={"segment": ["Basic", "Standard", "Premium"]})
fig.show()

# c) agregacja zamiast liczności: suma przychodu w binach ceny
fig = px.histogram(df, x="unit_price", y="revenue", histfunc="sum", nbins=30, title="Suma przychodu w przedziałach ceny")
fig.show()

# d) dystrybuanta (cumulative) i skala logarytmiczna
fig = px.histogram(df, x="revenue", cumulative=True, histnorm="probability", nbins=60, log_x=True, title="Skumulowany, log_x")
fig.show()

# e) zmienna dyskretna: słupki wycentrowane na wartościach
fig = px.histogram(df, x="satisfaction", text_auto=True, color_discrete_sequence=["#8172b3"])
fig.update_layout(bargap=0.15, xaxis=dict(tickmode="linear", dtick=1))
fig.show()

### 6.2. ECDF: `px.ecdf`

**Kiedy:** porównanie rozkładów **bez** wyboru binów, odczyt percentyli. **Interpretacja:** y = odsetek obserwacji ≤ x; krzywa bardziej w prawo/dół = wyższe wartości.

| Parametr | Opis |
|---|---|
| `x` (lub `y`) ✅ | |
| `color`, `facet_col` ⬜ | Grupy |
| `ecdfnorm` ⬜ | `"probability"` (domyślnie), `"percent"`, `None` (liczności) |
| `ecdfmode` ⬜ | `"standard"`, `"complementary"` (1 − ECDF), `"reversed"` |
| `markers`, `lines` ⬜ | Pokazywanie punktów / linii |
| `marginal` ⬜ | `"rug"` lub `"histogram"` |
| `log_x` ⬜ | |

In [73]:
fig = px.ecdf(df, x="revenue", color="segment", ecdfnorm="percent", marginal="rug",
              color_discrete_sequence=px.colors.sequential.Viridis[::3], category_orders={"segment": ["Basic", "Standard", "Premium"]},
              title="ECDF per segment + rug")
fig.add_hline(y=50, line_dash="dot", line_color="gray", annotation_text="mediana")
fig.add_hline(y=90, line_dash="dot", line_color="gray", annotation_text="P90")
fig.show()

fig = px.ecdf(df, x="delivery_days", color="channel", ecdfmode="complementary", ecdfnorm="percent",
              title="% dostaw trwających dłużej niż x dni")
fig.add_vline(x=5, line_dash="dash", line_color="red")
fig.show()

### 6.3. KDE i odpowiednik `distplot`

Plotly Express nie ma samodzielnego `kdeplot`, a `figure_factory.create_distplot` (histogram + KDE + rug, odpowiednik starego `sns.distplot`) **został usunięty w Plotly 7**. Zamiast niego buduj wykres z trzech elementów: `go.Histogram` (z `histnorm="probability density"`), krzywa KDE ze SciPy (`gaussian_kde`) i rug w osobnym wierszu `make_subplots`. Alternatywą jest `px.violin` (kształt gęstości w grupach, sekcja 7).

In [74]:
from scipy.stats import gaussian_kde

chs = ["Online", "Sklep", "Telefon"]
grid = np.linspace(0, df["delivery_days"].max(), 300)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.85, 0.15], vertical_spacing=0.02)
for k, ch_name in enumerate(chs):
    v = df.loc[df["channel"] == ch_name, "delivery_days"].dropna().to_numpy()
    col = cmap[ch_name]
    fig.add_trace(go.Histogram(x=v, histnorm="probability density", xbins=dict(size=0.5), name=ch_name,
                               marker_color=col, opacity=0.35, legendgroup=ch_name), row=1, col=1)          # histogram
    fig.add_trace(go.Scatter(x=grid, y=gaussian_kde(v, bw_method=0.3)(grid), mode="lines", line=dict(color=col, width=2.5),
                             name=f"KDE {ch_name}", legendgroup=ch_name, showlegend=False), row=1, col=1)   # KDE
    fig.add_trace(go.Scatter(x=v, y=np.full(len(v), k), mode="markers", legendgroup=ch_name, showlegend=False,
                             marker=dict(symbol="line-ns-open", color=col, size=10, opacity=0.4),
                             hoverinfo="skip"), row=2, col=1)                                               # rug
fig.update_layout(barmode="overlay", title="Histogram + KDE + rug (zamiennik create_distplot)")
fig.update_yaxes(title_text="Gęstość", row=1, col=1)
fig.update_yaxes(showticklabels=False, showgrid=False, row=2, col=1)
fig.update_xaxes(title_text="Dni dostawy", row=2, col=1)
fig.show()

# sama krzywa KDE z wypełnieniem (pełna kontrola, wiele grup)
fig = go.Figure()
for ch_name, col in cmap.items():
    v = df.loc[df["channel"] == ch_name, "delivery_days"].dropna()
    fig.add_trace(go.Scatter(x=grid, y=gaussian_kde(v, bw_method=0.3)(grid), name=ch_name, mode="lines",
                             fill="tozeroy", line=dict(color=col, width=2), opacity=0.5))
fig.update_layout(title="KDE: czas dostawy wg kanału", xaxis_title="Dni", yaxis_title="Gęstość")
fig.show()

> `bw_method` w `gaussian_kde` odpowiada `bw_adjust` w seaborn: mniejsza wartość = więcej szczegółów. Pamiętaj, że KDE wychodzi poza zakres danych (np. ujemne dni); przytnij zakres `grid` do sensownego minimum.

### 6.4. Rug

`px.histogram`, `px.ecdf`, `px.scatter` przyjmują `marginal="rug"`. Samodzielny rug: `go.Scatter` z `mode="markers"` i `marker_symbol="line-ns-open"`.

In [75]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=df["customer_age"], nbinsx=30, marker_color="#8da0cb", name="wiek", yaxis="y"))
fig.add_trace(go.Scatter(x=df["customer_age"], y=np.zeros(len(df)) - 3, mode="markers",
                         marker=dict(symbol="line-ns-open", color="black", size=8, opacity=0.3), name="rug", hoverinfo="skip"))
fig.update_layout(title="Histogram + własny rug", yaxis_title="Liczba")
fig.show()

## 7. Rozkłady w grupach: box, violin, strip, ridge

### 7.1. Boxplot: `px.box`

**Kiedy:** porównanie rozkładów wielu grup. **Interpretacja:** linia w pudełku = mediana; pudełko = IQR (Q1–Q3); wąsy = do 1,5 × IQR; punkty poza = potencjalne outliery. Zaleta Plotly: **najechanie pokazuje dokładne kwantyle** (min, Q1, mediana, Q3, max).

| Parametr | Opis |
|---|---|
| `x`, `y` ✅ | Kategoria + zmienna ciągła (zamień osie → poziomo) |
| `color` ⬜ | Podgrupy |
| `points` ⬜ | `"outliers"` (domyślnie), `"all"` (wszystkie punkty z jitterem), `"suspectedoutliers"`, `False` |
| `notched` ⬜ | Wcięcie = przybliżony CI mediany |
| `boxmode` ⬜ | `"group"` (obok siebie), `"overlay"` |
| `hover_data`, `hover_name` ⬜ | Dane w tooltipie dla punktów (np. `order_id`) |
| `facet_col`, `category_orders`, `color_discrete_map`, `log_y` ⬜ | |

`update_traces`: `boxmean=True` (znacznik średniej) lub `"sd"` (średnia + odch. std.), `jitter` (0–1), `pointpos` (przesunięcie punktów względem pudełka), `marker_size`, `line_width`, `whiskerwidth`, `quartilemethod` (`"linear"`, `"exclusive"`, `"inclusive"`).

In [76]:
order_seg = ["Basic", "Standard", "Premium"]

fig = px.box(df, x="segment", y="revenue", color="segment", points="outliers", notched=True,
             category_orders={"segment": order_seg}, color_discrete_sequence=px.colors.sequential.Viridis[::3],
             hover_data=["order_id"], title="Box: notched + outliery")
fig.update_traces(boxmean=True, line_width=2, marker=dict(size=4, opacity=0.5))
fig.update_layout(showlegend=False)
fig.show()

# grupowane (color) + wszystkie punkty (box + strip w jednym)
fig = px.box(smp, x="segment", y="delivery_days", color="channel", points="all", boxmode="group",
             category_orders={"segment": order_seg}, color_discrete_map=cmap, hover_name="order_id")
fig.update_traces(jitter=0.4, pointpos=0, marker_size=3, marker_opacity=0.5, fillcolor="rgba(255,255,255,0)")
fig.show()

# poziomy, bez outlierów, skala log
fig = px.box(df, y="segment", x="revenue", color="segment", points=False, log_x=True, category_orders={"segment": order_seg[::-1]})
fig.show()

### 7.2. Violinplot: `px.violin`

**Kiedy:** pełny kształt rozkładu w grupach (wielomodalność, skośność), próby ≥ 30–50 / grupę. **Interpretacja:** szerokość = gęstość (KDE); wewnątrz opcjonalnie mini-box.

| Parametr | Opis |
|---|---|
| `x`, `y` ✅ | Jak w box |
| `color` ⬜ | Podgrupy |
| `box` ⬜ | `True` = box wewnątrz |
| `points` ⬜ | `"all"`, `"outliers"`, `False` |
| `violinmode` ⬜ | `"group"`, `"overlay"` |
| `hover_data` ⬜ | |

`update_traces`: `meanline_visible=True`, `side` (`"both"`, `"positive"`, `"negative"`: połówki skrzypiec), `spanmode` (`"soft"` domyślnie rozszerza poza dane, `"hard"` przycina do zakresu = `cut=0` w seaborn), `bandwidth` (wygładzanie KDE), `scalemode` (`"width"`, `"count"`), `width`, `points`.

In [77]:
fig = px.violin(df, x="segment", y="revenue", color="segment", box=True, points=False,
                category_orders={"segment": order_seg}, color_discrete_sequence=px.colors.sequential.Viridis[::3])
fig.update_traces(spanmode="hard", meanline_visible=True, bandwidth=15)
fig.update_layout(showlegend=False, title="Violin: spanmode='hard', średnia")
fig.show()

# split violin: dwie grupy na jednym „skrzypku" (go.Violin + side)
fig = go.Figure()
for ret, side, col in [(True, "positive", "#2a9d8f"), (False, "negative", "#e76f51")]:
    d = df[df["is_returning"] == ret].dropna(subset=["delivery_days"])
    fig.add_trace(go.Violin(x=d["channel"], y=d["delivery_days"], side=side, name="Powracający" if ret else "Nowi",
                            line_color=col, fillcolor=col, opacity=0.6, meanline_visible=True, spanmode="hard", scalegroup="channel"))
fig.update_layout(violinmode="overlay", title="Split violin: nowi vs powracający", yaxis_title="Dni dostawy")
fig.show()

### 7.3. Strip (zamiast strip/swarm): `px.strip`

Plotly **nie ma swarmplot**. `px.strip` rysuje punkty z jitterem; zaletą jest interaktywność (hover na pojedynczych obserwacjach).

| Parametr | Opis |
|---|---|
| `x`, `y`, `color` | |
| `stripmode` ⬜ | `"group"` (obok siebie), `"overlay"` |
| `hover_name`, `hover_data` ⬜ | |

In [78]:
fig = px.strip(smp, x="segment", y="revenue", color="channel", stripmode="group", hover_name="order_id",
               category_orders={"segment": order_seg}, color_discrete_map=cmap, title="Strip")
fig.update_traces(marker=dict(size=5, opacity=0.6), jitter=0.8)
fig.show()

### 7.4. „Pointplot": średnie z przedziałami ufności

Agregacja + `error_y`. Równoległe linie = brak interakcji między czynnikami.

In [79]:
s = (df.groupby(["segment", "channel"], observed=True)["satisfaction"].agg(["mean", "std", "count"]).reset_index())
s["ci"] = 1.96 * s["std"] / np.sqrt(s["count"])
fig = px.line(s, x="segment", y="mean", color="channel", error_y="ci", markers=True, color_discrete_map=cmap,
              category_orders={"segment": order_seg}, title="Satysfakcja: segment × kanał (średnia ± 95% CI)")
fig.update_traces(marker_size=9)
fig.show()

### 7.5. „Countplot"

In [80]:
cnt = df["region"].value_counts().reset_index()               # kolumny: region, count
fig = px.bar(cnt, x="region", y="count", text_auto=True, color_discrete_sequence=["#4c72b0"], title="Liczba zamówień wg regionu")
fig.update_traces(textposition="outside")
fig.show()

fig = px.histogram(df, y="channel", color="segment", barmode="group", histnorm="percent", text_auto=".1f",
                   category_orders={"segment": order_seg}, title="Poziomy, % zamówień")        # histogram jako szybki countplot
fig.show()

### 7.6. Ridge plot (joyplot)

**Kiedy:** porównanie rozkładów ≥ 4–5 grup. **Interpretacja:** każda „fala" = gęstość jednej grupy na wspólnej osi X. W Plotly robi się to poziomymi, jednostronnymi violinami.

In [81]:
reg_order = df.groupby("region")["income"].median().sort_values().index.tolist()
colors = px.colors.sample_colorscale("Viridis", np.linspace(0.1, 0.9, len(reg_order)))

fig = go.Figure()
for reg, col in zip(reg_order, colors):
    fig.add_trace(go.Violin(x=df.loc[df["region"] == reg, "income"], name=reg, line_color="white", line_width=1.5,
                            fillcolor=col, opacity=0.85, meanline_visible=False))
fig.update_traces(orientation="h", side="positive", width=3, points=False)       # width > 1 = nakładanie fal
fig.update_layout(title="Ridge plot: dochód wg regionu", xaxis_showgrid=False, xaxis_zeroline=False, showlegend=False,
                  xaxis_title="Dochód", height=520)
fig.show()

## 8. Zależności między zmiennymi

### 8.1. Scatter: `px.scatter`

**Kiedy:** związek dwóch zmiennych ciągłych; do 5 wymiarów (x, y, kolor, rozmiar, kształt). **Interpretacja:** kierunek, siła, kształt, klastry, outliery. Korelacja ≠ przyczynowość. **Duże dane:** `render_mode="webgl"` (domyślnie od ~1000 punktów w `auto`) lub agregacja.

| Parametr | Opis |
|---|---|
| `x`, `y` ✅ | |
| `color` ⬜ | Kategoria lub liczba (skala ciągła) |
| `size`, `size_max` ⬜ | Rozmiar wg zmiennej (bubble), maksymalny rozmiar w px |
| `symbol` ⬜ | Kształt wg kategorii (`symbol_map`, `symbol_sequence`) |
| `opacity` ⬜ | Przezroczystość |
| `trendline` ⬜ | `"ols"`, `"lowess"`, `"rolling"`, `"expanding"`, `"ewm"` (OLS/lowess wymaga `statsmodels`) |
| `trendline_options` ⬜ | np. `dict(frac=0.3)` (lowess), `dict(window=5, function="median")` (rolling), `dict(log_x=True)` (ols potęgowy) |
| `trendline_scope` ⬜ | `"trace"` (osobna linia per grupa) lub `"overall"` |
| `trendline_color_override` ⬜ | Jeden kolor linii trendu |
| `marginal_x`, `marginal_y` ⬜ | `"histogram"`, `"rug"`, `"box"`, `"violin"` |
| `error_x`, `error_y` ⬜ | Błędy |
| `text`, `hover_name`, `hover_data`, `facet_col`, `log_x`, `log_y` ⬜ | |

In [82]:
# a) kolor kategoryczny + symbol
fig = px.scatter(smp, x="unit_price", y="revenue", color="segment", symbol="channel", opacity=0.7,
                 color_discrete_sequence=px.colors.sequential.Viridis[::3], category_orders={"segment": order_seg},
                 hover_name="order_id", hover_data={"discount": ":.1%"}, title="color + symbol")
fig.update_traces(marker=dict(size=8, line=dict(width=0.5, color="white")))
fig.show()

# b) kolor ciągły + rozmiar (bubble)
fig = px.scatter(smp, x="unit_price", y="revenue", color="discount", size="units", size_max=22,
                 color_continuous_scale="Plasma", opacity=0.7, title="Bubble: kolor = rabat, rozmiar = sztuki")
fig.update_coloraxes(colorbar_title="Rabat", colorbar_tickformat=".0%")
fig.show()

### 8.2. Linie trendu (odpowiednik `regplot` / `lmplot`)

In [83]:
# a) OLS dla całości + podsumowanie modelu
fig = px.scatter(smp, x="unit_price", y="revenue", trendline="ols", trendline_color_override="crimson",
                 opacity=0.5, title="OLS (całość)")
fig.show()
res = px.get_trendline_results(fig)
print(res.iloc[0]["px_fit_results"].summary().tables[1])         # współczynniki, p-value, CI

# b) osobny trend per grupa + panele (jak lmplot)
fig = px.scatter(smp, x="unit_price", y="revenue", color="segment", facet_col="channel", trendline="ols",
                 color_discrete_sequence=px.colors.sequential.Viridis[::3], category_orders={"segment": order_seg},
                 opacity=0.5, height=420, title="lmplot-style: trend per segment, panele per kanał")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))        # „channel=Online" → „Online"
fig.show()

# c) LOWESS (nieparametryczny) i trend potęgowy
fig = px.scatter(smp, x="customer_age", y="income", trendline="lowess", trendline_options=dict(frac=0.4),
                 trendline_color_override="purple", opacity=0.4, title="LOWESS")
fig.show()

# d) wielomian stopnia 2 ręcznie (np.polyfit) + pasmo ±2σ reszt
x, y = smp["unit_price"].to_numpy(), smp["revenue"].to_numpy()
coef = np.polyfit(x, y, 2)
xs = np.linspace(x.min(), x.max(), 200)
yhat = np.polyval(coef, xs)
sd = np.std(y - np.polyval(coef, x))
fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=y, mode="markers", marker=dict(color="#4c72b0", opacity=0.4), name="dane"))
fig.add_trace(go.Scatter(x=np.r_[xs, xs[::-1]], y=np.r_[yhat + 2 * sd, (yhat - 2 * sd)[::-1]], fill="toself",
                         fillcolor="rgba(220,20,60,0.12)", line=dict(width=0), name="±2σ reszt", hoverinfo="skip"))
fig.add_trace(go.Scatter(x=xs, y=yhat, mode="lines", line=dict(color="crimson", width=3), name="wielomian st. 2"))
fig.update_layout(title="Regresja wielomianowa + pasmo")
fig.show()

                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        -20.3784     16.085     -1.267      0.206     -51.968      11.212
x1             3.7297      0.131     28.413      0.000       3.472       3.987


**Regresja logistyczna i reszty** (statsmodels): Plotly nie ma natywnego `logistic=True`, więc liczysz model i rysujesz krzywą.

In [84]:
import statsmodels.api as sm

X = sm.add_constant(smp["income"].to_numpy())
logit = sm.Logit(smp["returning_int"].to_numpy(), X).fit(disp=0)
grid = np.linspace(smp["income"].min(), smp["income"].max(), 200)
prob = logit.predict(sm.add_constant(grid))

fig = go.Figure()
fig.add_trace(go.Scatter(x=smp["income"], y=smp["returning_int"] + rng.normal(0, 0.02, len(smp)), mode="markers",
                         marker=dict(opacity=0.35, size=6), name="obserwacje (jitter)"))
fig.add_trace(go.Scatter(x=grid, y=prob, mode="lines", line=dict(color="crimson", width=3), name="P(powracający)"))
fig.update_layout(title="Regresja logistyczna", yaxis_title="Prawdopodobieństwo", yaxis_tickformat=".0%")
fig.show()

# residplot: reszty z modelu liniowego
b1, b0 = np.polyfit(smp["unit_price"], smp["revenue"], 1)
resid = smp["revenue"] - (b1 * smp["unit_price"] + b0)
fig = px.scatter(x=smp["unit_price"], y=resid, labels={"x": "unit_price", "y": "reszta"}, opacity=0.5,
                 trendline="lowess", trendline_color_override="crimson", title="Reszty (lejek ⇒ heteroskedastyczność)")
fig.add_hline(y=0, line_dash="dash")
fig.show()

### 8.3. Odpowiednik `jointplot`: scatter + marginesy

In [85]:
fig = px.scatter(df, x="unit_price", y="revenue", color="segment", marginal_x="histogram", marginal_y="violin",
                 opacity=0.5, color_discrete_sequence=px.colors.sequential.Viridis[::3],
                 category_orders={"segment": order_seg}, title="Scatter + marginesy (jointplot)", height=600)
fig.update_traces(marker_size=5, selector=dict(type="scatter"))
fig.show()

## 9. Dwa wymiary (gęstość) i wykresy wielowymiarowe

### 9.1. Density heatmap i density contour (zamiast hexbin / histplot 2D / kde 2D)

Plotly **nie ma `hexbin` dla zwykłych osi X–Y** (dla map istnieje `figure_factory.create_hexbin_map`). Jego rolę (duże dane, overplotting) pełni `px.density_heatmap` (prostokątne biny) i `px.density_contour` (kontury).

| Parametr | Opis |
|---|---|
| `x`, `y` ✅ | |
| `z` ⬜ | Zmienna do agregacji w komórce (jak `C` w hexbin) |
| `histfunc` ⬜ | `"count"`, `"avg"`, `"sum"`, `"min"`, `"max"` (agregacja `z`) |
| `nbinsx`, `nbinsy` ⬜ | Liczba binów |
| `histnorm` ⬜ | Normalizacja |
| `marginal_x`, `marginal_y` ⬜ | `"histogram"`, `"box"`, `"violin"`, `"rug"` |
| `color_continuous_scale` ⬜ | Skala kolorów |
| `contours_coloring` ⬜ | (contour) `"fill"`, `"lines"`; `fig.update_traces(contours_coloring="fill", contours_showlabels=True)` (przy `marginal_*` dodaj `selector=dict(type="histogram2dcontour")`) |
| `facet_col`, `color` ⬜ | |

In [86]:
fig = px.density_heatmap(df, x="unit_price", y="revenue", nbinsx=40, nbinsy=40,
                         color_continuous_scale=px.colors.sequential.Viridis,         # z marginesami podaj skalę jako LISTĘ (nazwa tekstowa → błąd w Plotly 7.1)
                         marginal_x="histogram", marginal_y="histogram", title="Density heatmap: liczność")
fig.show()

fig = px.density_heatmap(df, x="unit_price", y="revenue", z="discount", histfunc="avg", nbinsx=25, nbinsy=25,
                         color_continuous_scale="RdYlGn_r", title="Średni rabat w komórkach (jak C w hexbin)")
fig.show()

fig = px.density_contour(df, x="unit_price", y="revenue", color="segment", marginal_x="box",
                         color_discrete_sequence=px.colors.sequential.Viridis[::3], title="Density contour per segment")
fig.update_traces(line_width=2, selector=dict(type="histogram2dcontour"))      # selector: tylko kontury (nie marginesy box)
fig.show()

fig = px.density_contour(df, x="unit_price", y="revenue", title="Contour wypełniony")
fig.update_traces(contours_coloring="fill", contours_showlabels=True, colorscale="Mint")
fig.show()

### 9.2. Scatter matrix (odpowiednik `pairplot`)

In [87]:
fig = px.scatter_matrix(smp, dimensions=["unit_price", "revenue", "delivery_days", "satisfaction"], color="segment",
                        color_discrete_sequence=px.colors.sequential.Viridis[::3], category_orders={"segment": order_seg},
                        opacity=0.5, height=720, title="Scatter matrix")
fig.update_traces(diagonal_visible=False, showupperhalf=False, marker_size=4)      # tylko dolny trójkąt (jak corner=True)
fig.show()

Przekątna w `scatter_matrix` jest puste (lub scatter): histogramy/KDE na przekątnej wymagają `go.Splom` + ręcznych dodatków, a to rzadko się opłaca. Do EDA rozkładów użyj osobnych `px.histogram`/`px.violin` z `facet_col`.

### 9.3. Wykresy wielowymiarowe: parallel coordinates, parallel categories, 3D, radar

In [88]:
# parallel coordinates: wiele zmiennych liczbowych, kolor wg jednej; przeciągnij zakres na osi, aby filtrować
fig = px.parallel_coordinates(smp, dimensions=["customer_age", "unit_price", "units", "discount", "revenue"],
                              color="revenue", color_continuous_scale="Viridis", title="Parallel coordinates")
fig.show()

# parallel categories: przepływ obserwacji przez kategorie
fig = px.parallel_categories(df, dimensions=["region", "channel", "segment"], color="satisfaction",
                             color_continuous_scale="RdYlGn", title="Parallel categories")
fig.show()

# 3D scatter
fig = px.scatter_3d(smp, x="unit_price", y="delivery_days", z="revenue", color="segment", opacity=0.7,
                    color_discrete_sequence=px.colors.sequential.Viridis[::3], category_orders={"segment": order_seg})
fig.update_traces(marker_size=3)
fig.show()

# radar: profil segmentów po kilku metrykach (znormalizowanych do 0–1)
met = df.groupby("segment", observed=True)[["unit_price", "units", "discount", "satisfaction", "delivery_days"]].mean()
met_n = (met - met.min()) / (met.max() - met.min())
long = met_n.reset_index().melt(id_vars="segment", var_name="metryka", value_name="wartość")
fig = px.line_polar(long, r="wartość", theta="metryka", color="segment", line_close=True,
                    color_discrete_sequence=px.colors.sequential.Viridis[::3], title="Radar: profil segmentów (min–max)")
fig.update_traces(fill="toself", opacity=0.5)
fig.show()

## 10. Macierze: heatmap i clustermap

### 10.1. Heatmap: `px.imshow` i `go.Heatmap`

**Kiedy:** macierz korelacji, tabela przestawna, kalendarz aktywności. **Interpretacja:** kolor = wartość. Dla korelacji paleta rozbieżna z `zmin=-1`, `zmax=1` (środek = 0).

| Parametr `px.imshow` | Opis |
|---|---|
| `img` ✅ | Macierz (DataFrame z etykietami lub `numpy`) |
| `text_auto` ⬜ | `True` lub format `".2f"`, `",.0f"`: liczby w komórkach |
| `color_continuous_scale` ⬜ | Skala (`"RdBu_r"`, `"YlGnBu"`, `"Viridis"`) |
| `zmin`, `zmax` ⬜ | Zakres kolorów |
| `color_continuous_midpoint` ⬜ | Środek skali rozbieżnej |
| `aspect` ⬜ | `"equal"` (kwadratowe komórki) lub `"auto"` (wypełnia wykres) |
| `labels` ⬜ | `dict(x="", y="", color="r")` |
| `origin` ⬜ | `"upper"` (domyślnie) lub `"lower"` |
| `x`, `y` ⬜ | Własne etykiety osi |

In [89]:
# a) korelacje: tylko dolny trójkąt (maska → NaN)
mask = np.triu(np.ones(corr.shape, dtype=bool), k=1)
fig = px.imshow(corr.mask(mask), text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                aspect="equal", labels=dict(color="Pearson r"), title="Macierz korelacji", height=650)
fig.update_xaxes(side="bottom", tickangle=-45)
fig.show()

# b) tabela przestawna region × segment
fig = px.imshow(pivot, text_auto=",.0f", color_continuous_scale="YlGnBu", aspect="auto",
                labels=dict(x="Segment", y="Region", color="Śr. przychód"), title="Region × segment")
fig.show()

# c) kalendarz: dzień tygodnia × miesiąc + obramowana komórka max (add_shape)
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
cal = (df.assign(m=df["order_date"].dt.month).pivot_table(index="weekday", columns="m", values="order_id", aggfunc="count").reindex(days))
fig = px.imshow(cal, text_auto=True, color_continuous_scale="Magma_r", aspect="auto", labels=dict(x="Miesiąc", y="", color="Zamówienia"))
r_i, c_i = np.unravel_index(np.nanargmax(cal.to_numpy()), cal.shape)
fig.add_shape(type="rect", x0=c_i - 0.5, x1=c_i + 0.5, y0=r_i - 0.5, y1=r_i + 0.5, line=dict(color="cyan", width=4))
fig.update_xaxes(dtick=1)
fig.show()

# d) go.Heatmap: odstępy między komórkami, własny hover, etykiety z dwiema wartościami
share = pivot.div(pivot.sum(axis=1), axis=0)
labels = pivot.round(0).astype(int).astype(str) + "<br>(" + (share * 100).round(0).astype(int).astype(str) + "%)"
fig = go.Figure(go.Heatmap(z=share.to_numpy(), x=[str(c) for c in share.columns], y=share.index.tolist(),
                           text=labels.to_numpy(), texttemplate="%{text}", colorscale="Teal", xgap=3, ygap=3,
                           colorbar=dict(title="Udział", tickformat=".0%"),
                           hovertemplate="Region: %{y}<br>Segment: %{x}<br>Udział: %{z:.1%}<extra></extra>"))
fig.update_layout(title="go.Heatmap: etykiety własne, odstępy", yaxis_autorange="reversed")
fig.show()

### 10.2. Clustermap (ręcznie: SciPy + `imshow`)

Plotly nie ma funkcji `clustermap`. Wzorzec: **kolejność z klastrowania hierarchicznego (SciPy) → przestawienie wierszy/kolumn → `px.imshow`**. Dendrogram osobno przez `ff.create_dendrogram`.

In [90]:
from scipy.cluster.hierarchy import linkage, leaves_list
import plotly.figure_factory as ff

# a) korelacje: przestaw zmienne tak, by podobne leżały obok siebie
Z = linkage(corr, method="average", metric="euclidean")
order = leaves_list(Z)
fig = px.imshow(corr.iloc[order, order], text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                aspect="equal", title="Clustermap (korelacje, kolejność z linkage)", height=650)
fig.show()

# b) obserwacje × cechy po standaryzacji (z-score kolumn): klastrowanie wierszy i kolumn
num_cols = ["customer_age", "income", "units", "unit_price", "discount", "revenue", "delivery_days", "satisfaction"]
sub = df.dropna(subset=num_cols).sample(60, random_state=5)
z = (sub[num_cols] - sub[num_cols].mean()) / sub[num_cols].std()
z.index = sub["order_id"] + " (" + sub["segment"].astype(str) + ")"                    # etykiety wierszy z segmentem

row_order = leaves_list(linkage(z, method="ward"))
col_order = leaves_list(linkage(z.T, method="ward"))
fig = px.imshow(z.iloc[row_order, col_order], color_continuous_scale="RdBu_r", zmin=-3, zmax=3, aspect="auto",
                labels=dict(color="z-score"), height=800, title="Clustermap obserwacji (ward, z-score)")
fig.update_yaxes(showticklabels=False)
fig.show()

# dendrogram jako osobny wykres
dend = ff.create_dendrogram(z.to_numpy()[:30], orientation="left", labels=[str(i) for i in range(30)])
dend.update_layout(title="Dendrogram (30 obserwacji)", height=600)
dend.show()

> Dla pełnoprawnego clustermap z dendrogramami po bokach w statycznym wykresie użyj `sns.clustermap`. W Plotly dendrogram + heatmapę da się złożyć przez `make_subplots`, ale to dużo kodu względem efektu.

## 11. Linie, cieniowanie, adnotacje, wyróżnianie elementów

### 11.1. Tabela metod

| Cel | Metoda | Ważne parametry |
|---|---|---|
| Pozioma / pionowa linia | `fig.add_hline(y=)`, `fig.add_vline(x=)` | `line_dash` (`"solid"`, `"dot"`, `"dash"`, `"longdash"`, `"dashdot"`), `line_color`, `line_width`, `opacity`, `annotation_text`, `annotation_position`, `row=`, `col=`, `layer="below"` |
| Pas poziomy / pionowy | `fig.add_hrect(y0=, y1=)`, `fig.add_vrect(x0=, x1=)` | `fillcolor`, `opacity`, `line_width=0`, `layer="below"`, `annotation_text` |
| Dowolna linia / prostokąt / elipsa | `fig.add_shape(type="line"/"rect"/"circle", x0, y0, x1, y1)` | `line=dict(color, width, dash)`, `fillcolor`, `xref`/`yref` (`"x"` = dane, `"paper"` = 0–1) |
| Wypełnienie pod/między liniami | `go.Scatter(fill="tozeroy"/"tonexty"/"toself")` | `fillcolor="rgba(r,g,b,a)"` |
| Tekst / strzałka | `fig.add_annotation(x, y, text)` | `showarrow`, `arrowhead` (1–7), `arrowcolor`, `ax`, `ay` (przesunięcie w px), `bgcolor`, `bordercolor`, `font`, `xanchor`, `yanchor` |
| Obrys zaznaczenia | `update_traces(selected=..., unselected=...)` | `marker=dict(opacity=, color=, size=)` |

`annotation_position`: `"top left"`, `"top right"`, `"bottom left"`, `"bottom right"`, `"top"`, `"bottom"` (hline/vline), dla rect także `"inside top left"` itp.

> **Uwaga:** `add_vline`/`add_vrect` z osią czasu (`datetime`) działają, ale `annotation_text` przy `add_vline` bywa kapryśny w zależności od wersji. Bezpieczny wzorzec: `add_vline` bez tekstu + osobne `add_annotation`.

### 11.2. Przykład: linie, pasy i adnotacje na szeregu czasowym

In [91]:
s = df.groupby("month", as_index=False)["revenue"].sum()
mean = s["revenue"].mean()

fig = px.line(s, x="month", y="revenue", markers=True, title="Linie, pasy, adnotacje")
fig.update_traces(line_color="#1f77b4", line_width=3)

fig.add_hline(y=mean, line_dash="dash", line_color="crimson", annotation_text=f"średnia {mean:,.0f}", annotation_position="top left")
fig.add_hrect(y0=mean * 0.9, y1=mean * 1.1, fillcolor="crimson", opacity=0.08, line_width=0, layer="below")     # pas ±10%
fig.add_vrect(x0="2023-11-01", x1="2024-01-31", fillcolor="orange", opacity=0.15, line_width=0, layer="below",
              annotation_text="Sezon", annotation_position="top left")
fig.add_vline(x=pd.Timestamp("2024-01-01"), line_dash="dot", line_color="gray")
fig.add_annotation(x=pd.Timestamp("2024-01-01"), y=1, yref="paper", text="Nowy rok", showarrow=False, yanchor="bottom", font=dict(color="gray"))

top = s.loc[s["revenue"].idxmax()]
fig.add_trace(go.Scatter(x=[top["month"]], y=[top["revenue"]], mode="markers+text", text=["Max"], textposition="top center",
                         marker=dict(size=16, color="gold", line=dict(color="black", width=2)), name="Maksimum", showlegend=False))
fig.show()

### 11.3. Wyróżnianie punktów, serii i zaznaczenia

In [92]:
# a) punkty: całość szara + wybrany podzbiór wyróżniony (z etykietami)
top5 = smp.nlargest(5, "revenue")
fig = px.scatter(smp, x="unit_price", y="revenue", opacity=0.5, color_discrete_sequence=["#c7cdd6"], hover_name="order_id")
fig.add_trace(go.Scatter(x=top5["unit_price"], y=top5["revenue"], mode="markers+text", text=top5["order_id"],
                         textposition="top center", marker=dict(size=13, color="#d62728", line=dict(color="black", width=1)), name="Top 5"))
fig.add_hline(y=smp["revenue"].quantile(0.95), line_dash="dash", annotation_text="P95")
fig.show()

# b) kolor warunkowy przez kolumnę pomocniczą (legenda działa automatycznie)
smp2 = smp.assign(grupa=np.where(smp["revenue"] > smp["revenue"].quantile(0.9), "Top 10%", "Reszta"))
fig = px.scatter(smp2, x="unit_price", y="revenue", color="grupa", color_discrete_map={"Top 10%": "#d62728", "Reszta": "#c7cdd6"},
                 category_orders={"grupa": ["Reszta", "Top 10%"]})                       # kolejność = kolejność rysowania (wyróżnione na wierzchu)
fig.show()

# c) wyróżnienie serii liniowej na tle pozostałych
pv = df.groupby(["month", "region"], as_index=False)["revenue"].sum()
fig = px.line(pv, x="month", y="revenue", color="region")
fig.update_traces(line=dict(color="lightgray", width=1.5))
fig.update_traces(line=dict(color="#d62728", width=4), selector=dict(name="Centrum"))
fig.update_layout(showlegend=False)
fig.add_annotation(x=pv["month"].max(), y=pv.loc[(pv["region"] == "Centrum") & (pv["month"] == pv["month"].max()), "revenue"].iloc[0],
                   text="Centrum", showarrow=False, xanchor="left", font=dict(color="#d62728"))
fig.show()

# d) styl zaznaczonych punktów (zaznacz prostokątem/lassem w interaktywnym wykresie)
fig = px.scatter(smp, x="unit_price", y="revenue", color_discrete_sequence=["#4c72b0"])
fig.update_traces(selected=dict(marker=dict(color="red", size=10)), unselected=dict(marker=dict(opacity=0.15)))
fig.update_layout(dragmode="select")
fig.show()

## 12. Układy złożone

### 12.1. Facety (small multiples)

| Parametr | Opis |
|---|---|
| `facet_col`, `facet_row` ⬜ | Zmienne dzielące na panele |
| `facet_col_wrap` ⬜ | Zawijanie (tylko z `facet_col`) |
| `facet_col_spacing`, `facet_row_spacing` ⬜ | Odstępy |
| `fig.update_yaxes(matches=None)` ⬜ | Niezależne osie Y w panelach (domyślnie wspólne) |
| `fig.update_xaxes(showticklabels=True)` ⬜ | Pokaż etykiety osi w każdym panelu |

In [93]:
fig = px.histogram(df, x="revenue", facet_col="region", facet_col_wrap=3, nbins=25, color_discrete_sequence=["#4c72b0"], height=550)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))        # czytelne tytuły paneli
fig.update_yaxes(matches=None, showticklabels=True)                             # niezależne osie Y
fig.add_vline(x=df["revenue"].median(), line_dash="dash", line_color="red")      # linia w każdym panelu
fig.show()

### 12.2. `make_subplots`: dowolny układ z różnych wykresów

In [94]:
fig = make_subplots(rows=2, cols=2, subplot_titles=("Trend", "Udział kanałów", "Box wg segmentu", "Histogram"),
                    specs=[[{"colspan": 2}, None],
                           [{"type": "xy"}, {"type": "xy"}]],
                    row_heights=[0.55, 0.45], vertical_spacing=0.14)

# wykresy z px można „przenieść": trace'y z figury px dodajemy do subplotu
f1 = px.line(df_month, x="month", y="revenue", color="channel", color_discrete_map=cmap)
for tr in f1.data:
    fig.add_trace(tr, row=1, col=1)

f2 = px.box(df, x="segment", y="revenue", color="segment", category_orders={"segment": order_seg},
            color_discrete_sequence=px.colors.sequential.Viridis[::3])
for tr in f2.data:
    tr.showlegend = False
    fig.add_trace(tr, row=2, col=1)

fig.add_trace(go.Histogram(x=df["revenue"], nbinsx=30, marker_color="#8da0cb", showlegend=False), row=2, col=2)

fig.update_xaxes(tickangle=-45, row=1, col=1)
fig.update_layout(height=750, title_text="make_subplots: układ z colspan", legend=dict(orientation="h", y=1.08))
fig.show()

| Parametr `make_subplots` | Opis |
|---|---|
| `rows`, `cols` ⬜ | Siatka |
| `specs` ⬜ | Lista list słowników: `{"colspan": 2}`, `{"rowspan": 2}`, `{"secondary_y": True}`, `{"type": "domain"}` (pie), `{"type": "polar"}`, `None` = pusta komórka |
| `subplot_titles` ⬜ | Tytuły |
| `shared_xaxes`, `shared_yaxes` ⬜ | `True`, `"rows"`, `"columns"` |
| `row_heights`, `column_widths` ⬜ | Proporcje |
| `vertical_spacing`, `horizontal_spacing` ⬜ | Odstępy |

### 12.3. Dual axis (druga oś Y)

In [95]:
tot = df.groupby("month").agg(revenue=("revenue", "sum"), orders=("order_id", "size")).reset_index()

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=tot["month"], y=tot["revenue"], name="Przychód", marker_color="#8da0cb", opacity=0.8), secondary_y=False)
fig.add_trace(go.Scatter(x=tot["month"], y=tot["orders"], name="Zamówienia", mode="lines+markers", line=dict(color="#d62728", width=3)),
              secondary_y=True)
fig.update_yaxes(title_text="Przychód [PLN]", tickformat=",.0f", title_font_color="#4c72b0", secondary_y=False)
fig.update_yaxes(title_text="Liczba zamówień", title_font_color="#d62728", showgrid=False, secondary_y=True)
fig.update_layout(title="Dual axis: przychód i zamówienia", hovermode="x unified", legend=dict(orientation="h", y=1.1))
fig.show()

### 12.4. KPI: `go.Indicator`

Przydatne w dashboardach: liczba, delta względem poprzedniego okresu, gauge.

In [96]:
cur, prev = tot["revenue"].iloc[-2], tot["revenue"].iloc[-3]
fig = make_subplots(rows=1, cols=2, specs=[[{"type": "indicator"}, {"type": "indicator"}]])
fig.add_trace(go.Indicator(mode="number+delta", value=cur, delta=dict(reference=prev, relative=True, valueformat=".1%"),
                           number=dict(valueformat=",.0f", suffix=" PLN"), title=dict(text="Przychód (ost. pełny mies.)")), row=1, col=1)
fig.add_trace(go.Indicator(mode="gauge+number", value=df["satisfaction"].mean(), number=dict(valueformat=".2f"),
                           title=dict(text="Śr. satysfakcja"),
                           gauge=dict(axis=dict(range=[1, 5]), bar=dict(color="#2a9d8f"),
                                      steps=[dict(range=[1, 3], color="#fde0dd"), dict(range=[3, 4], color="#fff7bc"), dict(range=[4, 5], color="#d9f0a3")],
                                      threshold=dict(line=dict(color="red", width=3), value=4))), row=1, col=2)
fig.update_layout(height=320)
fig.show()

### 12.5. Dashboard (mozaika różnych typów)

In [97]:
fig = make_subplots(rows=2, cols=3, specs=[[{"colspan": 2}, None, {"type": "domain"}],
                                          [{"type": "xy"}, {"type": "xy", "colspan": 2}, None]],
                    subplot_titles=("Trend wg kanału", "", "Udział kanałów", "Box wg segmentu", "Region × segment"),
                    row_heights=[0.5, 0.5], vertical_spacing=0.15)

for ch_name, g in df_month.groupby("channel"):
    fig.add_trace(go.Scatter(x=g["month"], y=g["revenue"], name=ch_name, mode="lines", line=dict(color=cmap[ch_name])), row=1, col=1)

ch = df.groupby("channel")["revenue"].sum()
fig.add_trace(go.Pie(labels=ch.index, values=ch.values, hole=0.5, marker=dict(colors=[cmap[c] for c in ch.index]), showlegend=False), row=1, col=3)

for seg_name, col in zip(order_seg, px.colors.sequential.Viridis[::3]):
    fig.add_trace(go.Box(y=df.loc[df["segment"] == seg_name, "revenue"], name=seg_name, marker_color=col, showlegend=False), row=2, col=1)

fig.add_trace(go.Heatmap(z=pivot.to_numpy(), x=[str(c) for c in pivot.columns], y=pivot.index.tolist(), colorscale="YlGnBu",
                         texttemplate="%{z:,.0f}", showscale=False), row=2, col=2)
fig.update_layout(height=700, title_text="<b>Dashboard sprzedaży</b>", legend=dict(orientation="h", y=1.08, x=0))
fig.show()

## 13. Interaktywność

### 13.1. Dropdown (przełączanie danych / typu wykresu)

`updatemenus` + `method`: `"update"` (zmiana danych i układu), `"restyle"` (tylko trace'y), `"relayout"` (tylko układ), `"animate"`.

In [98]:
metrics = {"Przychód": "revenue", "Sztuki": "units", "Rabat": "discount", "Dni dostawy": "delivery_days"}
fig = go.Figure()
for i, (label, col) in enumerate(metrics.items()):
    d = df.groupby("month")[col].mean().reset_index()
    fig.add_trace(go.Scatter(x=d["month"], y=d[col], mode="lines+markers", name=label, visible=(i == 0)))

buttons = [dict(label=label, method="update",
                args=[{"visible": [j == i for j in range(len(metrics))]},                  # widoczna tylko wybrana seria
                      {"title": f"Średnia miesięczna: {label}", "yaxis": {"title": label}}])
           for i, label in enumerate(metrics)]
fig.update_layout(updatemenus=[dict(buttons=buttons, direction="down", x=0.0, y=1.15, xanchor="left", showactive=True)],
                  title="Średnia miesięczna: Przychód", showlegend=False)
fig.show()

### 13.2. Animacja (`animation_frame`)

Dla wykresów punktowych **ustaw stałe zakresy osi**, inaczej wykres „skacze".

In [99]:
q = (df.groupby(["quarter", "region"], as_index=False)
       .agg(price=("unit_price", "mean"), satisf=("satisfaction", "mean"), orders=("order_id", "size"), revenue=("revenue", "sum")))
fig = px.scatter(q, x="price", y="satisf", size="orders", color="region", animation_frame="quarter", animation_group="region",
                 size_max=45, range_x=[q["price"].min() * 0.9, q["price"].max() * 1.1], range_y=[q["satisf"].min() * 0.95, q["satisf"].max() * 1.05],
                 hover_name="region", color_discrete_sequence=px.colors.qualitative.Set2, title="Regiony w czasie (kwartały)")
fig.update_layout(transition_duration=300)
fig.show()

# animowany słupkowy: stały zakres Y
fig = px.bar(q, x="region", y="revenue", color="region", animation_frame="quarter", range_y=[0, q["revenue"].max() * 1.1],
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.show()

### 13.3. Konfiguracja interakcji (`config`) i tryby przeciągania

In [100]:
fig = px.scatter(smp, x="unit_price", y="revenue", color="segment")
fig.update_layout(dragmode="zoom")                       # "zoom", "pan", "select", "lasso", "orbit" (3D), False
fig.show(config={
    "displaylogo": False,                                # ukryj logo Plotly
    "scrollZoom": True,                                  # zoom kółkiem myszy
    "displayModeBar": True,                              # pasek narzędzi: True / False / "hover"
    "modeBarButtonsToRemove": ["select2d", "lasso2d"],
    "toImageButtonOptions": {"format": "png", "filename": "wykres", "height": 600, "width": 1000, "scale": 2},   # przycisk „zapisz jako PNG"
})

### 13.4. Reakcja na kliknięcia (callbacki)

Plotly w notebooku jest interaktywny tylko w przeglądarce. Do logiki reagującej na kliknięcia/filtry potrzebny jest framework:

| Narzędzie | Kiedy |
|---|---|
| **Dash** (`dash`) | Pełne aplikacje webowe; callbacki `@app.callback`, Plotly natywnie |
| **Streamlit** (`st.plotly_chart(fig)`) | Najszybszy dashboard w Pythonie (opcja `on_select="rerun"` zwraca zaznaczone punkty) |
| `go.FigureWidget` (+ `ipywidgets`) | Callbacki w samym notebooku (`trace.on_click(...)`) |

## 14. Mapy

Plotly (≥ 5.24) ma mapy oparte na MapLibre: `px.scatter_map`, `px.density_map`, `px.choropleth_map`, `px.line_map` (starsze `*_mapbox` są przestarzałe). Style bez tokenu: `"open-street-map"`, `"carto-positron"`, `"carto-darkmatter"`, `"white-bg"`.

| Parametr `px.scatter_map` | Opis |
|---|---|
| `lat`, `lon` ✅ | Kolumny współrzędnych (WGS84) |
| `color`, `size`, `size_max`, `hover_name`, `hover_data` ⬜ | Jak w scatter |
| `zoom`, `center` ⬜ | Zoom (≈ 5 = kraj) i `dict(lat=, lon=)` |
| `map_style` ⬜ | Styl podkładu |
| `opacity`, `animation_frame` ⬜ | |

In [101]:
cities = pd.DataFrame({
    "miasto": ["Warszawa", "Kraków", "Gdańsk", "Wrocław", "Poznań", "Łódź"],
    "lat": [52.23, 50.06, 54.35, 51.11, 52.41, 51.76], "lon": [21.01, 19.94, 18.65, 17.04, 16.93, 19.46],
    "przychod": [980, 640, 520, 580, 510, 430]})

fig = px.scatter_map(cities, lat="lat", lon="lon", size="przychod", color="przychod", hover_name="miasto", size_max=35, zoom=5,
                     center=dict(lat=52.0, lon=19.4), map_style="carto-positron", color_continuous_scale="Viridis", height=550)
fig.show()

# choropleth z GeoJSON (np. z GeoPandas: geojson = gdf.__geo_interface__ — gdf musi mieć EPSG:4326 i kolumnę id)
geojson = {"type": "FeatureCollection", "features": [
    {"type": "Feature", "id": "A", "properties": {"id": "A"}, "geometry": {"type": "Polygon", "coordinates": [[[19, 51], [21, 51], [21, 52], [19, 52], [19, 51]]]}},
    {"type": "Feature", "id": "B", "properties": {"id": "B"}, "geometry": {"type": "Polygon", "coordinates": [[[21, 51], [23, 51], [23, 52], [21, 52], [21, 51]]]}}]}
vals = pd.DataFrame({"id": ["A", "B"], "wartosc": [120, 80]})
fig = px.choropleth_map(vals, geojson=geojson, locations="id", featureidkey="properties.id", color="wartosc",
                        color_continuous_scale="YlOrRd", zoom=5.5, center=dict(lat=51.5, lon=21), map_style="carto-positron", opacity=0.6, height=500)
fig.show()

## 15. Zapis, wydajność, pułapki

### 15.1. Zapis

In [102]:
fig = px.bar(df.groupby("region", as_index=False)["revenue"].sum(), x="region", y="revenue")

fig.write_html("wykres.html", include_plotlyjs="cdn", full_html=True)       # interaktywny plik do udostępnienia
# include_plotlyjs: True (plik ~4 MB, działa offline), "cdn" (mały plik, wymaga internetu), "directory"

# statyczne obrazy wymagają pakietu kaleido (pip install kaleido) i przeglądarki Chrome/Chromium
# fig.write_image("wykres.png", width=1000, height=600, scale=2)            # png, jpg, svg, pdf, webp
# fig.to_json()  /  pio.write_json(fig, "wykres.json")                      # zapis definicji wykresu
fig.show()

| Metoda | Opis |
|---|---|
| `write_html(file, include_plotlyjs, full_html, config)` | `full_html=False` zwraca sam fragment `<div>` do wstawienia w stronę |
| `write_image(file, format, width, height, scale)` | Wymaga `kaleido`; `scale=2` = wyższa rozdzielczość |
| `to_html()`, `to_json()`, `to_dict()` | Reprezentacje do dalszego przetwarzania |

### 15.2. Wydajność

| Problem | Rozwiązanie |
|---|---|
| Scatter z 50k+ punktów wolny | `render_mode="webgl"` (lub `go.Scattergl`), próbkowanie, `density_heatmap`; dla milionów punktów `datashader` |
| Wykres z wielu serii/panelu zamula | Zmniejsz liczbę trace'ów (agreguj), unikaj `facet` z setkami paneli |
| Duży plik HTML | `include_plotlyjs="cdn"`, mniej punktów (każdy punkt jest w pliku!) |
| Duże dane w SQL Server/BigQuery | Agreguj po stronie SQL/DuckDB/Polars i rysuj tylko wynik |

> Plotly **przechowuje wszystkie dane w figurze** (i w wyeksportowanym HTML). Wykres dla 1 mln wierszy to wielomegabajtowy plik, a seaborn trzyma dane tylko w pamięci Pythona. Dlatego agregacja przed rysowaniem jest ważniejsza niż w matplotlib.

### 15.3. Typowe pułapki

1. **`px.bar` nie agreguje.** Surowe dane dają nałożone słupki. Użyj `groupby` albo `px.histogram(histfunc="sum")`.
2. **Kolor „ciągły" zamiast „kategorycznego":** kolumna liczbowa o kilku wartościach (`satisfaction`) dostaje skalę ciągłą. Zamień na tekst (`.astype(str)`) i ustaw `category_orders`.
3. **Kolejność kategorii:** użyj `category_orders={"kol": [...]}`; kolejność w legendzie i na osi wynika z kolejności w danych.
4. **`update_traces` bez `selector`** zmienia wszystkie serie, a z `selector` tylko wybrane.
5. **Facety i etykiety:** tytuły paneli mają postać `kol=wartość`. Usuń prefiks przez `for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))`.
6. **Daty jako tekst:** konwertuj do `datetime64`, inaczej oś jest kategoryczna (a nie `date`).
7. **Hover z dużą liczbą pól** spowalnia i przeszkadza. Ogranicz `hover_data`, wyłącz niepotrzebne (`{"kol": False}`).
8. **Animacje:** ustaw stałe `range_x`/`range_y` i `animation_group`, żeby elementy się „płynnie" przesuwały.
9. **Wyświetlanie w VS Code/Jupyter:** gdy wykres się nie pokazuje, ustaw `pio.renderers.default` (`"vscode"`, `"notebook_connected"`, `"browser"`) i upewnij się, że `nbformat` jest zainstalowany.
10. **Power BI:** wizualizacje Python w Power BI zwracają statyczny obraz (matplotlib/seaborn); interaktywny Plotly tam nie działa. Do raportów BI zostaw natywne wizualizacje, a Plotly przydaje się w notebooku, Streamlit lub Dash.

## 16. Ściągawka: seaborn → Plotly

| Cel | seaborn | Plotly Express |
|---|---|---|
| Linia z CI | `lineplot` (CI automatyczny) | `px.line` na zagregowanych + `error_y` lub pasmo `go.Scatter(fill="tonexty")` |
| Słupki | `barplot`, `countplot` | `px.bar` (agreguj sam), `px.histogram` dla liczności |
| Histogram | `histplot`, `displot(kind="hist")` | `px.histogram` (`marginal`, `histnorm`, `cumulative`) |
| KDE | `kdeplot` | `scipy.gaussian_kde` + `go.Scatter` (+ `go.Histogram`), lub violin |
| ECDF | `ecdfplot` | `px.ecdf` |
| Rug | `rugplot` | `marginal="rug"` |
| Box / boxen | `boxplot`, `boxenplot` | `px.box` (boxen: brak) |
| Violin | `violinplot` | `px.violin` (`side=` dla split) |
| Strip / swarm | `stripplot`, `swarmplot` | `px.strip` (swarm: brak) |
| Pointplot | `pointplot` | `px.line` + `error_y` |
| Scatter | `scatterplot` | `px.scatter` |
| Regresja | `regplot`, `lmplot` | `px.scatter(trendline="ols"/"lowess")` + `facet_col` |
| Joint | `jointplot` | `px.scatter(marginal_x=, marginal_y=)` |
| Pairplot | `pairplot` | `px.scatter_matrix` |
| Hexbin / 2D hist / 2D KDE | `jointplot(kind="hex")`, `histplot(x, y)`, `kdeplot(x, y)` | `px.density_heatmap`, `px.density_contour` |
| Heatmap | `heatmap` | `px.imshow`, `go.Heatmap` |
| Clustermap | `clustermap` | SciPy `linkage` + `px.imshow` |
| Ridge | `FacetGrid` + `kdeplot` | `go.Violin(side="positive", orientation="h", width=3)` |
| Panele | `FacetGrid`, `col=` | `facet_col`, `facet_row` |
| Pie | (matplotlib) | `px.pie` |
| Dodatkowe w Plotly | – | `sunburst`, `treemap`, `funnel`, `Waterfall`, `Sankey`, `Indicator`, `parallel_coordinates`, `scatter_3d`, mapy |

### Kiedy Plotly, a kiedy seaborn?

| Sytuacja | Wybór |
|---|---|
| Eksploracja z hover, zoom, filtrowanie | **Plotly** |
| Dashboard / aplikacja (Streamlit, Dash), raport HTML | **Plotly** |
| Raport PDF, publikacja, Power BI Python visual | **seaborn/matplotlib** (statyczne, pełna kontrola typografii) |
| Statystyczne podsumowania z CI (lineplot/barplot z bootstrapem), ridge, clustermap | **seaborn** (wbudowane) |
| Bardzo duże dane (> 100k punktów) | Agregacja + dowolna; Plotly wymaga WebGL/Datashader |
| Wykresy nietypowe (Sankey, waterfall, funnel, 3D, mapy) | **Plotly** |

## Co dalej (rozwój w kierunku Twojego stacku)

- **Dash / Streamlit** na Plotly: interaktywny dashboard nad SQL Server lub BigQuery (zapytanie → pandas/Polars → `px`).
- **DuckDB + Plotly:** `duckdb.sql("SELECT ... GROUP BY ...").df()` → `px.bar(...)`: agregacja w silniku kolumnowym, rysowanie tylko wyniku.
- **GeoPandas + Plotly:** `gdf.to_crs(4326)`, `gdf.__geo_interface__` → `px.choropleth_map` (interaktywny odpowiednik map z QGIS).
- **Airflow:** raport HTML z Plotly (`write_html`) jako wynik zadania DAG-a, wysyłany mailem lub publikowany.
- **Plotly + scikit-learn:** interaktywne krzywe ROC, macierze pomyłek (`px.imshow`), ważność cech (`px.bar`), rzutowanie PCA/t-SNE (`px.scatter`, `px.scatter_3d`).